# 🔘📁 Gradio Multiple Tabs

⚠️ **Remember to copy this notebook in your Drive and rename.**

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**🗒️ Gradio's [Documentation](https://www.gradio.app/docs)**

## Setup

In [1]:
import gradio as gr
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import cv2

In [2]:
# Global variable to store the current image across tabs
current_image = None

## Create Functions

In [3]:
#Functions for various image processing, these will be linked to your buttons and various tabs

def basic_edit(image, brightness, contrast, blur):
    """Basic image editing: brightness, contrast, blur"""
    global current_image

    if image is None:
        return None, "Please upload an image first", None

    # Convert to PIL Image if it's a numpy array
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    # Apply brightness
    if brightness != 1.0:
        enhancer = ImageEnhance.Brightness(image)
        image = enhancer.enhance(brightness)

    # Apply contrast
    if contrast != 1.0:
        enhancer = ImageEnhance.Contrast(image)
        image = enhancer.enhance(contrast)

    # Apply blur
    if blur > 0:
        image = image.filter(ImageFilter.GaussianBlur(radius=blur))

    # Store the edited image globally
    current_image = image

    return image, f"Basic editing applied! Image saved for further editing.", image

def color_edit(hue_shift, saturation, warmth):
    """Color editing: hue, saturation, warmth"""
    global current_image

    if current_image is None:
        return None, "Please edit an image in the Basic Edit tab first", None

    image = current_image.copy()

    # Convert to numpy array for OpenCV operations
    img_array = np.array(image)

    # Convert RGB to HSV for hue and saturation adjustments
    hsv = cv2.cvtColor(img_array, cv2.COLOR_RGB2HSV).astype(np.float32)

    # Adjust hue
    if hue_shift != 0:
        hsv[:, :, 0] = (hsv[:, :, 0] + hue_shift) % 180

    # Adjust saturation
    if saturation != 1.0:
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation, 0, 255)

    # Convert back to RGB
    img_array = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # Apply warmth (adjust red and blue channels)
    if warmth != 0:
        img_array = img_array.astype(np.float32)
        if warmth > 0:  # Warmer (more red/yellow)
            img_array[:, :, 0] = np.clip(img_array[:, :, 0] * (1 + warmth * 0.3), 0, 255)
            img_array[:, :, 1] = np.clip(img_array[:, :, 1] * (1 + warmth * 0.1), 0, 255)
            img_array[:, :, 2] = np.clip(img_array[:, :, 2] * (1 - warmth * 0.2), 0, 255)
        else:  # Cooler (more blue)
            img_array[:, :, 0] = np.clip(img_array[:, :, 0] * (1 + warmth * 0.2), 0, 255)
            img_array[:, :, 1] = np.clip(img_array[:, :, 1] * (1 + warmth * 0.1), 0, 255)
            img_array[:, :, 2] = np.clip(img_array[:, :, 2] * (1 - warmth * 0.3), 0, 255)
        img_array = img_array.astype(np.uint8)

    result_image = Image.fromarray(img_array)

    # Update the global image
    current_image = result_image

    return result_image, "Color adjustments applied!", result_image

def advanced_edit(sharpness, noise_reduction, edge_enhance):
    """Advanced editing: sharpness, noise reduction, edge enhancement"""
    global current_image

    if current_image is None:
        return None, "Please edit an image in previous tabs first", None

    image = current_image.copy()

    # Apply sharpness
    if sharpness != 1.0:
        enhancer = ImageEnhance.Sharpness(image)
        image = enhancer.enhance(sharpness)

    # Apply noise reduction (simple blur)
    if noise_reduction > 0:
        image = image.filter(ImageFilter.GaussianBlur(radius=noise_reduction * 0.5))

    # Apply edge enhancement
    if edge_enhance:
        image = image.filter(ImageFilter.EDGE_ENHANCE)

    # Update the global image
    current_image = image

    return image, "Advanced editing applied!", image

def reset_image():
    """Reset the current image"""
    global current_image
    current_image = None
    return None, "Image reset. Please upload a new image in the Basic Edit tab."

def get_current_image():
    """Get the current image for viewing"""
    global current_image
    if current_image is None:
        return None, "No image available. Please edit an image in other tabs first."
    return current_image, "Current edited image"

## Gradio Interface

In [6]:
# change theme to whichever template
with gr.Blocks(title="Multi-Tab Image Editor", theme=gr.themes.Glass()) as demo:
    gr.Markdown("#  Multi-Tab Image Editor")
    gr.Markdown("Edit images across multiple tabs with different tools. Start by uploading an image in the **Basic Edit** tab, then move to other tabs for more editing options.")

    with gr.Tabs():
        # Basic Edit Tab
        with gr.TabItem("🔧 Basic Edit"):
            gr.Markdown("### Upload and apply basic edits to your image")

            with gr.Row():
                with gr.Column():
                    image_input = gr.Image(type="pil", label="Upload Image")
                    brightness_slider = gr.Slider(0.1, 2.0, value=1.0, step=0.1, label="Brightness")
                    contrast_slider = gr.Slider(0.1, 2.0, value=1.0, step=0.1, label="Contrast")
                    blur_slider = gr.Slider(0, 5, value=0, step=0.5, label="Blur")
                    basic_edit_btn = gr.Button("Apply Basic Edits", variant="primary")

                with gr.Column():
                    basic_output = gr.Image(label="Edited Image")
                    basic_status = gr.Textbox(label="Status", interactive=False)

        # Color Edit Tab
        with gr.TabItem(" Color Edit"):
            gr.Markdown("### Adjust colors of your image")
            gr.Markdown("*Note: You need to edit an image in the Basic Edit tab first*")

            with gr.Row():
                with gr.Column():
                    color_preview = gr.Image(label="Current Image Preview", interactive=False)
                    hue_slider = gr.Slider(-180, 180, value=0, step=10, label="Hue Shift")
                    saturation_slider = gr.Slider(0.0, 2.0, value=1.0, step=0.1, label="Saturation")
                    warmth_slider = gr.Slider(-1.0, 1.0, value=0.0, step=0.1, label="Warmth (Negative=Cool, Positive=Warm)")
                    color_edit_btn = gr.Button("Apply Color Edits", variant="primary")

                with gr.Column():
                    color_output = gr.Image(label="Color-Edited Image")
                    color_status = gr.Textbox(label="Status", interactive=False)

        # Advanced Edit Tab
        with gr.TabItem(" Advanced Edit"):
            gr.Markdown("### Apply advanced effects")
            gr.Markdown("*Note: This works on the image from previous tabs*")

            with gr.Row():
                with gr.Column():
                    advanced_preview = gr.Image(label="Current Image Preview", interactive=False)
                    sharpness_slider = gr.Slider(0.0, 3.0, value=1.0, step=0.1, label="Sharpness")
                    noise_slider = gr.Slider(0, 3, value=0, step=0.5, label="Noise Reduction")
                    edge_checkbox = gr.Checkbox(label="Edge Enhancement", value=False)
                    advanced_edit_btn = gr.Button("Apply Advanced Edits", variant="primary")

                with gr.Column():
                    advanced_output = gr.Image(label="Final Image")
                    advanced_status = gr.Textbox(label="Status", interactive=False)

        # View Final Result Tab
        with gr.TabItem(" View Result"):
            gr.Markdown("### View your final edited image")

            with gr.Row():
                with gr.Column():
                    view_btn = gr.Button("Show Current Image", variant="primary")
                    reset_btn = gr.Button("Reset All", variant="stop")

                with gr.Column():
                    final_output = gr.Image(label="Final Result")
                    final_status = gr.Textbox(label="Status", interactive=False)

            view_btn.click(
                get_current_image,
                outputs=[final_output, final_status]
            )

            reset_btn.click(
                reset_image,
                outputs=[final_status]
            )

    # Connect the editing functions to update previews in next tabs
    basic_edit_btn.click(
        basic_edit,
        inputs=[image_input, brightness_slider, contrast_slider, blur_slider],
        outputs=[basic_output, basic_status, color_preview]
    )

    color_edit_btn.click(
        color_edit,
        inputs=[hue_slider, saturation_slider, warmth_slider],
        outputs=[color_output, color_status, advanced_preview]
    )

    advanced_edit_btn.click(
        advanced_edit,
        inputs=[sharpness_slider, noise_slider, edge_checkbox],
        outputs=[advanced_output, advanced_status, final_output]
    )

    gr.Markdown("""
    ## How to Use:
    1. **Basic Edit**: Upload an image and adjust brightness, contrast, and blur
    2. **Color Edit**: Modify hue, saturation, and warmth of your image
    3. **Advanced Edit**: Apply sharpness, noise reduction, and edge enhancement
    4. **View Result**: See your final edited image or reset everything

    The image is automatically passed between tabs
    """)

demo.launch(share=True, prevent_thread_lock=True)

/tmp/ipykernel_10326/491819997.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Multi-Tab Image Editor", theme=gr.themes.Glass()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70f7aefb96943f0195.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
